Day 0:Preparation & Data

In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
import time

geolocator = Nominatim(user_agent="internship_route_optimization")

addresses = [
    "Intramuros, Manila, Philippines",                  
    "Roxas Blvd, Manila, Philippines",                  
    "Quezon Memorial Circle, Quezon City, Philippines", 
    "UP Town Center, Quezon City, Philippines",         
    "BGC, Taguig, Philippines",                         
    "Ayala Triangle, Makati, Philippines"               
]

coords = []
print("Geocoding addresses... please wait.")

for addr in addresses:
    try:
        loc = geolocator.geocode(addr, timeout=10)
        if loc:
            coords.append((addr, loc.latitude, loc.longitude))
            print(f"Success: {addr}")
        time.sleep(1) 
    except Exception as e:
        print(f"Error geocoding {addr}: {e}")

df_points = pd.DataFrame(coords, columns=['place', 'lat', 'lon'])

print("\n--- Deliverable: df_points ---")
print(df_points)

Geocoding addresses... please wait.
Success: Intramuros, Manila, Philippines
Success: Roxas Blvd, Manila, Philippines
Success: Quezon Memorial Circle, Quezon City, Philippines
Success: UP Town Center, Quezon City, Philippines
Success: BGC, Taguig, Philippines
Success: Ayala Triangle, Makati, Philippines

--- Deliverable: df_points ---
                                              place        lat         lon
0                   Intramuros, Manila, Philippines  14.590959  120.974654
1                   Roxas Blvd, Manila, Philippines  14.579672  120.977254
2  Quezon Memorial Circle, Quezon City, Philippines  14.651447  121.049288
3          UP Town Center, Quezon City, Philippines  14.649864  121.074827
4                          BGC, Taguig, Philippines  14.545398  121.045997
5               Ayala Triangle, Makati, Philippines  14.557241  121.024862


Day 1:  Build road network graph with OSMnx (NetworkX graph)

In [2]:
import osmnx as ox
import networkx as nx
from shapely.geometry import Point

center_point = (14.590959, 120.974654) 

print("Downloading road network... this may take a minute.")
G = ox.graph_from_point(center_point, dist=15000, network_type='drive')

G_proj = ox.project_graph(G)

node_ids = ox.nearest_nodes(G, df_points['lon'], df_points['lat'])

df_points['node'] = node_ids

print("\n--- Deliverable: G & Updated df_points ---")
print(f"Graph nodes: {len(G.nodes)}")
print(f"Graph edges: {len(G.edges)}")
print(df_points[['place', 'node']])


--- Deliverable: G & Updated df_points ---
Graph nodes: 48234
Graph edges: 119179
                                              place        node
0                   Intramuros, Manila, Philippines   163723137
1                   Roxas Blvd, Manila, Philippines  3202651965
2  Quezon Memorial Circle, Quezon City, Philippines    17216338
3          UP Town Center, Quezon City, Philippines    32520021
4                          BGC, Taguig, Philippines   148846566
5               Ayala Triangle, Makati, Philippines   371022105


Day 2: Query OSRM for travel times/distances and build a cost matrix

In [4]:
import requests
import numpy as np
import pandas as pd

OSRM_BASE = "http://router.project-osrm.org"  

coords_str = ";".join([f"{lon},{lat}" for lat, lon in zip(df_points['lat'], df_points['lon'])])

print("Querying OSRM Table service...")
url = f"{OSRM_BASE}/table/v1/driving/{coords_str}?annotations=duration,distance"
resp = requests.get(url)
resp.raise_for_status()
table = resp.json()

durations = np.array(table['durations'])
distances = np.array(table['distances'])

durations_df = pd.DataFrame(durations, index=df_points.place, columns=df_points.place)

print("\n--- Deliverable: Durations Matrix (Seconds) ---")
print(durations_df.round(0)) 

Querying OSRM Table service...

--- Deliverable: Durations Matrix (Seconds) ---
place                                             Intramuros, Manila, Philippines  \
place                                                                               
Intramuros, Manila, Philippines                                               0.0   
Roxas Blvd, Manila, Philippines                                             236.0   
Quezon Memorial Circle, Quezon City, Philippines                           1110.0   
UP Town Center, Quezon City, Philippines                                   1398.0   
BGC, Taguig, Philippines                                                   1010.0   
Ayala Triangle, Makati, Philippines                                         801.0   

place                                             Roxas Blvd, Manila, Philippines  \
place                                                                               
Intramuros, Manila, Philippines                                      

Day 3: Route optimization (heuristic / Clarke-Wright savings)

In [5]:
import math
from collections import defaultdict

df_points = df_points.reset_index(drop=True)
n = len(df_points)
depot = 0

savings = []
for i in range(1, n):
    for j in range(i + 1, n):
        s = durations[depot][i] + durations[depot][j] - durations[i][j]
        savings.append((s, i, j))

savings.sort(reverse=True, key=lambda x: x[0])

routes = {i: [depot, i, depot] for i in range(1, n)}
route_of = {i: i for i in range(1, n)}

for s, i, j in savings:
    ri = route_of.get(i)
    rj = route_of.get(j)
    
    if ri is None or rj is None or ri == rj:
        continue
        
    route_i = routes[ri]
    route_j = routes[rj]
    
    if route_i[-2] == i and route_j[1] == j:
        new_route = route_i[:-1] + route_j[1:]
    elif route_j[-2] == j and route_i[1] == i:
        new_route = route_j[:-1] + route_i[1:]
    else:
        continue
        
    new_route_id = min(ri, rj)
    routes[new_route_id] = new_route
    del routes[max(ri, rj)]
    
    for node in new_route:
        if node != depot:
            route_of[node] = new_route_id

final_routes = list(routes.values())
readable_routes = []
for rt in final_routes:
    readable_routes.append([df_points.loc[idx, 'place'] for idx in rt])

print("\n--- Deliverable: Optimized Routes ---")
for i, route in enumerate(readable_routes):
    print(f"Route {i+1}: {' -> '.join(route)}")


--- Deliverable: Optimized Routes ---
Route 1: Intramuros, Manila, Philippines -> Quezon Memorial Circle, Quezon City, Philippines -> UP Town Center, Quezon City, Philippines -> BGC, Taguig, Philippines -> Ayala Triangle, Makati, Philippines -> Roxas Blvd, Manila, Philippines -> Intramuros, Manila, Philippines


Day 4: Evaluate & refine routes, integrate OSRM route geometry for visualization

In [12]:
import json
from shapely.geometry import LineString, shape

routes_geo = []

print("Requesting route geometries from OSRM...")

for rt in final_routes:
    node_idxs = rt  
    coords = []
    for idx in node_idxs:
        lat = df_points.loc[idx, 'lat']
        lon = df_points.loc[idx, 'lon']
        coords.append(f"{lon},{lat}")
    
    coord_str = ";".join(coords)
    
    url = f"{OSRM_BASE}/route/v1/driving/{coord_str}?overview=full&geometries=geojson&steps=false"
    r = requests.get(url)
    r.raise_for_status()
    rr = r.json()
    
    if 'routes' in rr and len(rr['routes']) > 0:
        geom = rr['routes'][0]['geometry']  
        duration = rr['routes'][0]['duration'] 
        distance = rr['routes'][0]['distance'] 
        
        routes_geo.append({
            'route_nodes': node_idxs, 
            'geometry': geom, 
            'duration': duration, 
            'distance': distance
        })
    else:
        routes_geo.append({
            'route_nodes': node_idxs, 
            'geometry': None, 
            'duration': None, 
            'distance': None
        })

print("\n--- Deliverable: routes_geo ---")
print(f"Retrieved geometry for {len(routes_geo)} route(s).")
if routes_geo[0]['geometry']:
    print(f"Total Travel Distance: {routes_geo[0]['distance']/1000:.2f} km")
    print(f"Total Estimated Time: {routes_geo[0]['duration']/60:.2f} minutes")

Requesting route geometries from OSRM...

--- Deliverable: routes_geo ---
Retrieved geometry for 1 route(s).
Total Travel Distance: 50.12 km
Total Estimated Time: 69.30 minutes


Day 5: Visualization with Folium & final deliverables

In [14]:
import folium
from branca.colormap import linear
from folium.features import GeoJson

center = [df_points.lat.mean(), df_points.lon.mean()]
m = folium.Map(location=center, zoom_start=11)

for i, row in df_points.iterrows():
    if i == depot:
        folium.Marker(
            [row.lat, row.lon], 
            tooltip=row.place, 
            icon=folium.Icon(color='red', icon='home')
        ).add_to(m)
    else:
        folium.CircleMarker(
            [row.lat, row.lon], 
            radius=6, 
            tooltip=row.place, 
            color='blue', 
            fill=True,
            fill_opacity=0.7
        ).add_to(m)

colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 'darkgreen']

for idx, r in enumerate(routes_geo):
    geom = r['geometry']
    if geom:
        folium.GeoJson(
            geom,
            style_function=lambda feat, c=colors[idx % len(colors)]: {
                'color': c, 
                'weight': 5, 
                'opacity': 0.8
            },
            tooltip=f"Route {idx+1} - {r['duration']/60:.1f} min"
        ).add_to(m)

m.save("routes_map.html")

m

Data Export

In [16]:
import numpy as np
import json

np.save("durations.npy", durations)
print("Successfully saved durations.npy")

with open("routes_geo.json", "w") as f:
    json.dump(routes_geo, f, indent=4)
print("Successfully saved routes_geo.json")

import os
print(f"\nFiles saved in: {os.getcwd()}")
print("- durations.npy")
print("- routes_geo.json")
print("- routes_map.html")

Successfully saved durations.npy
Successfully saved routes_geo.json

Files saved in: C:\Users\JERMAGNE
- durations.npy
- routes_geo.json
- routes_map.html
